# 🎨 Тема 4. Раскрашиваем чёрно-белое видео

**Часть 1** — модель Lab: яркость отдельно, цвет отдельно.
**Часть 2** — данные: кадры мультфильма и аугментация.
**Часть 3** — автоэнкодер на PyTorch и **шаг обучения**.
**Часть 4** — раскрашиваем кадры, которых сеть не видела, и сохраняем видео.
**Часть 5** — КТ: раскрашиваем своё видео.

**Как устроено каждое задание**
1. Короткое объяснение и пример.
2. **Задание ✏️** — замените `None` на свой код и запустите (**Shift + Enter**).
3. **🧪 Проверка** — запустите ячейку под заданием: ✅ или ❌ и подсказка.
4. Застряли — **💡 Подсказка**, потом **🆘 Почти готовый ответ**.

> Сначала: **Файл → Сохранить копию на Диске**. Шпаргалка — в конце.

In [ ]:
# Библиотеки и учебное видео (качает сервер Colab, а не ваш интернет)
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from google.colab.patches import cv2_imshow

!wget -q -nc https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/Megamind.avi
torch.manual_seed(0)
np.random.seed(0)

cap = cv2.VideoCapture("Megamind.avi")
cap.set(cv2.CAP_PROP_POS_FRAMES, 100)
ok, img = cap.read()
cap.release()
img = cv2.resize(img, (360, 264))
print("Кадр:", img.shape)
cv2_imshow(img)

In [ ]:
#@title ▶️ Запустите: помощники для видео { display-mode: "form" }
# show_video(кадры)    — показать список кадров плеером
# record_webcam(5)     — записать 5 секунд с веб-камеры (браузер спросит разрешение)
# read_video(путь)     — прочитать видео в список кадров
import base64, subprocess
from IPython.display import HTML, Javascript, display

def show_video(frames, fps=12, width=480):
    if len(frames) == 0 or any(f is None for f in frames):
        print("⚠️ Показывать нечего: список пуст или в нём None")
        return
    prep = []
    for f in frames:
        f = np.clip(np.asarray(f), 0, 255).astype(np.uint8)
        if f.ndim == 2:
            f = cv2.cvtColor(f, cv2.COLOR_GRAY2BGR)
        h, w = f.shape[:2]
        prep.append(np.ascontiguousarray(f[: h - h % 2, : w - w % 2]))
    h, w = prep[0].shape[:2]
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "bgr24", "-s", f"{w}x{h}",
           "-r", str(fps), "-i", "-", "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "28",
           "-movflags", "+faststart", "-f", "mp4", "-"]
    data = subprocess.run(cmd, input=b"".join(f.tobytes() for f in prep), capture_output=True).stdout
    b64 = base64.b64encode(data).decode()
    display(HTML(f'<video width="{width}" controls autoplay loop muted><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

def record_webcam(seconds=5, filename="webcam.webm"):
    from google.colab.output import eval_js
    display(Javascript("""
    async function recordVideo(ms) {
      const stream = await navigator.mediaDevices.getUserMedia({video: true, audio: false});
      const video = document.createElement('video');
      video.srcObject = stream; video.muted = true; video.width = 320;
      document.body.appendChild(video);
      await video.play();
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      const recorder = new MediaRecorder(stream, {mimeType: 'video/webm'});
      const chunks = [];
      recorder.ondataavailable = e => chunks.push(e.data);
      const stopped = new Promise(resolve => recorder.onstop = resolve);
      recorder.start();
      await new Promise(resolve => setTimeout(resolve, ms));
      recorder.stop(); await stopped;
      stream.getTracks().forEach(t => t.stop()); video.remove();
      const bytes = new Uint8Array(await new Blob(chunks).arrayBuffer());
      let binary = '';
      for (let i = 0; i < bytes.length; i += 0x8000) binary += String.fromCharCode.apply(null, bytes.subarray(i, i + 0x8000));
      return btoa(binary);
    }"""))
    print(f"Запись {seconds} с… Подвигайтесь, покажите в кадр что-нибудь цветное!")
    data = eval_js(f"recordVideo({int(seconds * 1000)})")
    open(filename, "wb").write(base64.b64decode(data))
    fixed = "webcam.mp4"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", filename, "-r", "12", "-an", fixed], check=True)
    return fixed

def read_video(path, max_frames=300, size=(128, 96)):
    cap = cv2.VideoCapture(path)
    frames = []
    while len(frames) < max_frames:
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(cv2.resize(frame, size))
    cap.release()
    print("Прочитано кадров:", len(frames))
    return frames

print("✅ Помощники готовы: show_video, record_webcam, read_video")

---
# Часть 1. Модель Lab: яркость отдельно, цвет отдельно

В BGR цвет и яркость перемешаны в трёх числах. В модели **Lab** они разделены:
* **L** — яркость. Это и есть чёрно-белая картинка.
* **a** — ось «зелёный ↔ красный».
* **b** — ось «синий ↔ жёлтый».

В OpenCV все три числа от 0 до 255, а **128 у `a` и `b` значит «цвета нет»**.

Чёрно-белый фильм — это только L. Чтобы его раскрасить, нужно угадать **всего два числа на пиксель**: a и b.

### ✏️ Задание 1. Переведите кадр `img` в Lab: переменная `lab`

In [ ]:
lab = None

In [ ]:
#@title 🧪 Проверка задания 1 { display-mode: "form" }
def _check(tests):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}")
            print(f"   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание 1 выполнено! Пройдено тестов: {len(tests)} из {len(tests)}")
    return True

if _check([
    ('перевод выполнен', lambda: lab is not None and lab.shape == img.shape, 'cv2.cvtColor(img, cv2.COLOR_BGR2LAB)'),
    ('это Lab, а не исходный кадр', lambda: not np.array_equal(lab, img), 'Нужен код перевода cv2.COLOR_BGR2LAB'),
]):
    print("Пиксель в центре: L, a, b =", lab[132, 180])

<details><summary>💡 Подсказка</summary>

Как перевод в серый или HSV, только код `cv2.COLOR_BGR2LAB`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
lab = cv2.cvtColor(img, cv2.___)
```
</details>

### ✏️ Задание 2. Разложите `lab` на три канала: `L`, `a`, `b`

**Что должно получиться:** L выглядит как обычное чёрно-белое фото, а a и b — почти серые с пятнами там, где есть цвет.

In [ ]:
L, a, b = None, None, None

In [ ]:
#@title 🧪 Проверка задания 2 { display-mode: "form" }
def _check(tests):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}")
            print(f"   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание 2 выполнено! Пройдено тестов: {len(tests)} из {len(tests)}")
    return True

if _check([
    ('каналы получены', lambda: L is not None and L.shape == img.shape[:2], 'cv2.split(lab) возвращает три канала'),
    ('L — первый канал', lambda: np.array_equal(L, lab[:, :, 0]), 'Порядок: L, a, b'),
]):
    cv2_imshow(np.hstack([L, a, b]))
    print("Слева L (яркость), в центре a, справа b")

<details><summary>💡 Подсказка</summary>

`cv2.split` — как для каналов B, G, R в теме 2.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
L, a, b = cv2.split(___)
```
</details>

### ✏️ Задание 3. Узнайте a и b у серого пикселя: картинка `gray_img` залита цветом (128, 128, 128)

In [ ]:
gray_img = np.full((10, 10, 3), 128, dtype=np.uint8)
gray_lab = None
gray_a, gray_b = None, None   # a и b любого пикселя

In [ ]:
#@title 🧪 Проверка задания 3 { display-mode: "form" }
def _check(tests):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}")
            print(f"   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание 3 выполнено! Пройдено тестов: {len(tests)} из {len(tests)}")
    return True

if _check([
    ('перевод в Lab', lambda: gray_lab is not None and gray_lab.shape == (10, 10, 3), 'cv2.cvtColor(gray_img, cv2.COLOR_BGR2LAB)'),
    ('a = 128', lambda: int(gray_a) == 128, 'a — канал номер 1: gray_lab[0, 0, 1]'),
    ('b = 128', lambda: int(gray_b) == 128, 'b — канал номер 2: gray_lab[0, 0, 2]'),
]):
    print("У серого a =", gray_a, "и b =", gray_b, "— цвета нет")

<details><summary>💡 Подсказка</summary>

Пиксель `[0, 0]`, каналы номер 1 и 2.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
gray_a, gray_b = gray_lab[0, 0, ___], gray_lab[0, 0, ___]
```
</details>

### ✏️ Задание 4. Сотрите цвет: замените a и b на 128, соберите Lab обратно и переведите в BGR (`no_color`)

**Что должно получиться:** справа чёрно-белая версия кадра. Наша нейросеть будет делать обратное: по L угадывать a и b.

In [ ]:
flat = np.full_like(L, 128)                   # канал, где везде 128
lab_gray = None      # L оставляем, a и b — 128
no_color = None

In [ ]:
#@title 🧪 Проверка задания 4 { display-mode: "form" }
def _check(tests):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}")
            print(f"   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание 4 выполнено! Пройдено тестов: {len(tests)} из {len(tests)}")
    return True

if _check([
    ('Lab собран', lambda: lab_gray is not None and lab_gray.shape == img.shape, 'cv2.merge([L, flat, flat])'),
    ('переведено в BGR', lambda: no_color is not None and no_color.shape == img.shape, 'cv2.cvtColor(lab_gray, cv2.COLOR_LAB2BGR)'),
    ('цвета не осталось (B ≈ G ≈ R)', lambda: np.abs(no_color[:, :, 0].astype(int) - no_color[:, :, 2]).max() <= 3, 'a и b должны быть flat (128)'),
]):
    cv2_imshow(np.hstack([img, no_color]))

<details><summary>💡 Подсказка</summary>

`cv2.merge` склеивает каналы обратно. Обратный перевод — `cv2.COLOR_LAB2BGR`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
lab_gray = cv2.merge([L, flat, ___])
no_color = cv2.cvtColor(lab_gray, cv2.COLOR_LAB2___)
```
</details>

---
# Часть 2. Данные и аугментация

Учить будем на кадрах самого мультфильма: **вход — L, ответ — a и b**. Первые 200 кадров для обучения, остальные 70 — проверка: их сеть не увидит.

Кадры уменьшаем до 128×96: так обучение займёт меньше минуты.

In [ ]:
frames = read_video("Megamind.avi", size=(128, 96))
train_frames, test_frames = frames[:200], frames[200:]
print("Обучение:", len(train_frames), "кадров | проверка:", len(test_frames), "кадров")
show_video(test_frames)

Нейросети удобнее числа около нуля. Поэтому:
* **L** делим на 255 → от 0 до 1;
* **a, b**: вычитаем 128 и делим на 128 → от −1 до 1 (0 — «цвета нет»).

### ✏️ Задание 5. Нормализуйте a и b: `ab_norm = (ab - 128) / 128`

In [ ]:
ab = lab[:, :, 1:].astype(np.float32)        # каналы a и b кадра img
ab_norm = None

In [ ]:
#@title 🧪 Проверка задания 5 { display-mode: "form" }
def _check(tests):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}")
            print(f"   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание 5 выполнено! Пройдено тестов: {len(tests)} из {len(tests)}")
    return True

if _check([
    ('посчитано', lambda: ab_norm is not None and ab_norm.shape == ab.shape, 'Формула из условия: (ab - 128) / 128'),
    ('числа от −1 до 1', lambda: ab_norm.min() >= -1 and ab_norm.max() <= 1, 'Вычтите 128 и разделите на 128'),
    ('128 превратилось в 0', lambda: abs(((np.float32(128) - 128) / 128)) < 1e-6 and np.allclose(ab_norm, (ab - 128) / 128), 'Именно (ab - 128) / 128'),
]):
    print("min =", ab_norm.min().round(2), " max =", ab_norm.max().round(2))

<details><summary>💡 Подсказка</summary>

Сначала вычитание, потом деление — не забудьте скобки.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
ab_norm = (ab - ___) / ___
```
</details>

In [ ]:
# Перевод списка кадров в тензоры PyTorch: L — вход сети, ab — правильный ответ
def to_tensors(imgs):
    lab = np.stack([cv2.cvtColor(i, cv2.COLOR_BGR2LAB) for i in imgs]).astype(np.float32)
    L = torch.from_numpy(lab[..., :1] / 255).permute(0, 3, 1, 2)            # (N, 1, H, W)
    ab = torch.from_numpy((lab[..., 1:] - 128) / 128).permute(0, 3, 1, 2)   # (N, 2, H, W)
    return L, ab

L_test, ab_test = to_tensors(test_frames)
print("L_test:", tuple(L_test.shape), " ab_test:", tuple(ab_test.shape))

**Аугментация** — маленькие случайные изменения картинки при обучении: сдвиг, поворот, увеличение, уменьшение, отражение. Сеть видит каждый кадр «по-новому» и учит общие правила («кожа — оранжевая»), а не запоминает конкретные кадры.

`cv2.getRotationMatrix2D(центр, угол, масштаб)` готовит поворот с масштабом, `cv2.warpAffine` его применяет. Сдвиг дописываем в последний столбец матрицы.

### ✏️ Задание 6. Допишите функцию аугментации: поворот −10…10°, масштаб 0.9…1.15, сдвиг до 8 пикселей

**Что должно получиться:** пять разных вариантов одного кадра: чуть повёрнуты, сдвинуты, увеличены.

In [ ]:
def augment(frame):
    h, w = frame.shape[:2]
    angle = np.random.uniform(-10, 10)        # поворот в градусах
    scale = 1.0                              # ← случайный масштаб 0.9…1.15
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)
    M[:, 2] += np.random.uniform(-8, 8, 2)    # сдвиг по x и y
    out = frame.copy()   # ← cv2.warpAffine(frame, M, (w, h), borderMode=cv2.BORDER_REFLECT)
    if np.random.rand() < 0.5:
        out = cv2.flip(out, 1)                # зеркало
    return out

In [ ]:
#@title 🧪 Проверка задания 6 { display-mode: "form" }
def _check(tests):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}")
            print(f"   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание 6 выполнено! Пройдено тестов: {len(tests)} из {len(tests)}")
    return True

if _check([
    ('размер не меняется', lambda: augment(train_frames[100]).shape == train_frames[100].shape, 'warpAffine(frame, M, (w, h)) — размер (ширина, высота)'),
    ('картинка меняется', lambda: all(not np.array_equal(augment(train_frames[100]), train_frames[100]) for _ in range(8)), 'Примените cv2.warpAffine вместо frame.copy()'),
]):
    cv2_imshow(cv2.resize(np.hstack([augment(train_frames[100]) for _ in range(5)]), None, fx=1.5, fy=1.5))

<details><summary>💡 Подсказка</summary>

Масштаб — как угол, только `np.random.uniform(0.9, 1.15)`. Вторая строка — подсказка в комментарии справа.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
scale = np.random.uniform(0.9, ___)
out = cv2.warpAffine(frame, M, (w, h), borderMode=cv2.BORDER_REFLECT)
```
</details>

---
# Часть 3. Автоэнкодер и шаг обучения

**Автоэнкодер** — сеть из двух половин:
* **энкодер** сжимает картинку в маленький «код» (свёртки с шагом 2 уменьшают кадр в 2 раза каждая);
* **декодер** разворачивает код обратно в картинку нужного размера.

Сжатие заставляет сеть оставить только главное: «где лицо, где фон, где волосы». У нас вход — L (1 канал), выход — a и b (2 канала).

In [ ]:
class ColorNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(                                  # 96×128 → 12×16
            nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU())
        self.decoder = nn.Sequential(                                  # 12×16 → 96×128
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 2, 4, stride=2, padding=1), nn.Tanh())   # Tanh: ответ от −1 до 1

    def forward(self, x):                      # forward — как данные идут через сеть
        return self.decoder(self.encoder(x))

model = ColorNet()
print(model)

### ✏️ Задание 7. Посмотрите, во что энкодер сжимает один кадр: форма кода — в `code_shape`

In [ ]:
code = None         # прогоните через model.encoder первый кадр L_test[:1]
code_shape = None          # tuple(code.shape)

In [ ]:
#@title 🧪 Проверка задания 7 { display-mode: "form" }
def _check(tests):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}")
            print(f"   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание 7 выполнено! Пройдено тестов: {len(tests)} из {len(tests)}")
    return True

if _check([
    ('код получен', lambda: code is not None and code.ndim == 4, 'model.encoder(L_test[:1])'),
    ('форма (1, 128, 12, 16)', lambda: code_shape == (1, 128, 12, 16), 'code_shape = tuple(code.shape)'),
]):
    print("Вход:", tuple(L_test[:1].shape), "→ код:", code_shape)
    print("Пространство 96×128 сжато в 12×16, зато 128 «признаков» на каждую клетку")

<details><summary>💡 Подсказка</summary>

Энкодер — это `model.encoder`, его можно вызвать отдельно, как функцию.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
code = model.encoder(L_test[:___])
code_shape = tuple(code.___)
```
</details>

### ✏️ Задание 8. Сделайте прямой проход (forward) для всех проверочных кадров: `pred = model(L_test)`

In [ ]:
with torch.no_grad():                    # без запоминания градиентов: только смотрим
    pred = None

In [ ]:
#@title 🧪 Проверка задания 8 { display-mode: "form" }
def _check(tests):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}")
            print(f"   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание 8 выполнено! Пройдено тестов: {len(tests)} из {len(tests)}")
    return True

if _check([
    ('предсказание получено', lambda: pred is not None, 'pred = model(L_test)'),
    ('форма как у ab_test', lambda: tuple(pred.shape) == tuple(ab_test.shape), 'На выходе 2 канала (a, b) того же размера'),
]):
    print("pred:", tuple(pred.shape), "| числа от", round(pred.min().item(), 2), "до", round(pred.max().item(), 2))
    print("Сеть ещё не обучена — отвечает случайно")

<details><summary>💡 Подсказка</summary>

Модель вызывается как функция: `model(вход)`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
pred = model(___)
```
</details>

### Шаг обучения — главное в теме
Одна итерация обучения — всегда четыре действия:
1. **forward**: `pred = model(L)` — сеть даёт ответ;
2. **loss**: `loss = loss_fn(pred, ab)` — насколько ответ хуже правильного (MSE — средний квадрат ошибки);
3. **backward**: `opt.zero_grad()` и `loss.backward()` — PyTorch считает, куда подвинуть каждый вес;
4. **step**: `opt.step()` — оптимизатор двигает веса.

Ниже весь цикл обучения уже написан, кроме самого шага. Допишите 4 строчки.

### ✏️ Задание 9. Допишите шаг обучения и обучите сеть (20 эпох, около минуты)

**Что должно получиться:** ошибка на проверке падает с ~0.007 до ~0.002. Это кадры, которых сеть не видела.

In [ ]:
torch.manual_seed(0); np.random.seed(0)
model = ColorNet()
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
loss_fn = nn.MSELoss()
USE_AUGMENT = True     # 🔬 потом попробуйте False и сравните ошибку на проверке
history = []

for epoch in range(20):
    order = np.random.permutation(len(train_frames))
    for i in range(0, len(order), 16):                       # пачки по 16 кадров
        batch = [train_frames[j] for j in order[i:i + 16]]
        if USE_AUGMENT:
            batch = [augment(f) for f in batch]
        L, ab = to_tensors(batch)
        # 1) pred = ...   2) loss = ...   3) opt.zero_grad() и loss.backward()   4) opt.step()
        pass
    with torch.no_grad():
        test_loss = loss_fn(model(L_test), ab_test).item()
    history.append(test_loss)
    print(f"эпоха {epoch + 1:2d}: ошибка на проверке {test_loss:.4f}")

In [ ]:
#@title 🧪 Проверка задания 9 { display-mode: "form" }
def _check(tests):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}")
            print(f"   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание 9 выполнено! Пройдено тестов: {len(tests)} из {len(tests)}")
    return True

if _check([
    ('обучение прошло 20 эпох', lambda: len(history) == 20, 'Запустите ячейку целиком'),
    ('ошибка падает', lambda: history[-1] < history[0], 'Проверьте все 4 строчки шага: без opt.step() веса не меняются'),
    ('сеть лучше «серой» картинки', lambda: history[-1] < 0.5 * loss_fn(torch.zeros_like(ab_test), ab_test).item(), 'Обучение должно снизить ошибку хотя бы вдвое по сравнению с ответом «цвета нет»'),
]):
    gray_loss = loss_fn(torch.zeros_like(ab_test), ab_test).item()
    print(f"Ответ «цвета нет»: {gray_loss:.4f}  |  наша сеть: {history[-1]:.4f}")

<details><summary>💡 Подсказка</summary>

Четыре строчки в порядке: forward → loss → zero_grad + backward → step.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
pred = model(L)
loss = loss_fn(pred, ___)
opt.zero_grad()
loss.___()
opt.___()
```
</details>

---
# Часть 4. Раскрашиваем кадры, которых сеть не видела

Берём у кадра только яркость L, сеть угадывает a и b, склеиваем и переводим обратно в BGR. Обратная нормализация: `ab * 128 + 128`.

### ✏️ Задание 10. Раскрасьте проверочные кадры: список `colored`

**Что должно получиться:** лицо и фон раскрашены правдоподобно. Синий свитер станет бурым: сеть «усредняет» цвета, которые видела редко.

In [ ]:
with torch.no_grad():
    pred = model(L_test)
L_np = (L_test.permute(0, 2, 3, 1).numpy() * 255)            # (N, H, W, 1) — обратно в 0…255
ab_np = None   # ← pred.permute(0, 2, 3, 1).numpy() * 128 + 128
lab_pred = np.concatenate([L_np, ab_np], axis=-1).clip(0, 255).astype(np.uint8)
colored = [None for x in lab_pred]   # ← каждый x из Lab в BGR

In [ ]:
#@title 🧪 Проверка задания 10 { display-mode: "form" }
def _check(tests):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}")
            print(f"   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание 10 выполнено! Пройдено тестов: {len(tests)} из {len(tests)}")
    return True

if _check([
    ('a и b вернули в 0…255', lambda: ab_np is not None and ab_np.shape[-1] == 2 and ab_np.min() >= 0, 'pred.permute(0, 2, 3, 1).numpy() * 128 + 128'),
    ('кадров столько же', lambda: len(colored) == len(test_frames), 'Список по всем x из lab_pred'),
    ('кадры цветные', lambda: all(c is not None and c.shape == test_frames[0].shape for c in colored) and np.abs(colored[0][:, :, 0].astype(int) - colored[0][:, :, 2]).mean() > 5, 'cv2.cvtColor(x, cv2.COLOR_LAB2BGR)'),
]):
    gray3 = [cv2.cvtColor(cv2.cvtColor(f, cv2.COLOR_BGR2GRAY), cv2.COLOR_GRAY2BGR) for f in test_frames]
    print('Слева чёрно-белое, в центре наша раскраска, справа оригинал')
    show_video([np.hstack([g, c, f]) for g, c, f in zip(gray3, colored, test_frames)], width=640)

<details><summary>💡 Подсказка</summary>

Обратная нормализация к `(ab - 128) / 128` — умножить на 128 и прибавить 128.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
ab_np = pred.permute(0, 2, 3, 1).numpy() * ___ + ___
colored = [cv2.cvtColor(x, cv2.COLOR_LAB2___) for x in lab_pred]
```
</details>

### ✏️ Задание 11. Сохраните `colored` в `video/colorized.mp4` (FPS = 12, размер `(128, 96)`)

In [ ]:
os.makedirs("video", exist_ok=True)
out = None
# цикл out.write(...) и out.release()

In [ ]:
#@title 🧪 Проверка задания 11 { display-mode: "form" }
def _check(tests):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}")
            print(f"   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание 11 выполнено! Пройдено тестов: {len(tests)} из {len(tests)}")
    return True

if _check([
    ('файл создан', lambda: os.path.exists('video/colorized.mp4') and os.path.getsize('video/colorized.mp4') > 1000, 'VideoWriter, цикл write, release'),
    ('все кадры записаны', lambda: int(cv2.VideoCapture('video/colorized.mp4').get(cv2.CAP_PROP_FRAME_COUNT)) == len(colored), 'Размер в VideoWriter — (128, 96): (ширина, высота)'),
]):
    print("video/colorized.mp4:", os.path.getsize("video/colorized.mp4") // 1024, "КБ")

<details><summary>💡 Подсказка</summary>

Как в теме 3: `VideoWriter(имя, fourcc, fps, (ширина, высота))`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
out = cv2.VideoWriter("video/colorized.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 12, (128, 96))
for f in colored:
    out.___(f)
out.___()
```
</details>

### 🔬 Эксперимент: зачем аугментация
Поставьте в задании 9 `USE_AUGMENT = False`, перезапустите обучение и сравните ошибку на проверке.

**Мой ответ** (дважды щёлкните): стала ошибка на проверке больше или меньше? Почему?

…

---
# Часть 5. КТ — раскрасить своё видео (2 балла LXP)

Задание LXP «Раскрасить своё собственное видео»: обучить сеть на кадрах своего ролика **с аугментацией** (сдвиги, уменьшение, увеличение, поворот) и раскрасить отрывок.

1. Запишите 5–10 секунд с веб-камеры (`SOURCE = "webcam"`) или загрузите свой файл (`"upload"`). Покажите в кадр что-нибудь цветное.
2. Первые 70% кадров — обучение, остальные 30% — раскраска.
3. Сохраните `video/my_colorized.mp4` — слева чёрно-белое, справа раскраска.

**Сдать:** ссылку на ноутбук и файл `video/my_colorized.mp4`.

In [ ]:
SOURCE = "webcam"  # @param ["webcam", "upload"]

if SOURCE == "webcam":
    my_frames = read_video(record_webcam(8))
else:
    from google.colab import files
    uploaded = files.upload()
    my_frames = read_video(list(uploaded)[0])

split = int(len(my_frames) * 0.7)
my_train, my_test = my_frames[:split], my_frames[split:]
print("обучение:", len(my_train), "кадров | раскраска:", len(my_test))

In [ ]:
# КТ: обучите НОВУЮ сеть на my_train (задание 9: цикл с augment и шагом обучения)
# и раскрасьте my_test (задание 10). Затем сохраните video/my_colorized.mp4 (задание 11),
# склеив кадры: np.hstack([чёрно-белый, раскрашенный]) — размер (256, 96).

---
# Шпаргалка
| Что | Код |
|---|---|
| в Lab и обратно | `cv2.cvtColor(img, cv2.COLOR_BGR2LAB)` · `cv2.COLOR_LAB2BGR` |
| каналы | `L, a, b = cv2.split(lab)` · `cv2.merge([L, a, b])` |
| «цвета нет» | a = b = 128 |
| нормализация | `L / 255` · `(ab - 128) / 128` и обратно `ab * 128 + 128` |
| аугментация | `M = cv2.getRotationMatrix2D(центр, угол, масштаб)` → `cv2.warpAffine(img, M, (w, h))` |
| forward | `pred = model(L)` |
| шаг обучения | `loss = loss_fn(pred, ab)` → `opt.zero_grad()` → `loss.backward()` → `opt.step()` |
| без обучения | `with torch.no_grad(): ...` |
| тензор → numpy | `t.permute(0, 2, 3, 1).numpy()` |